<a href="https://colab.research.google.com/github/WARRAICH-11/NETSOL/blob/main/Project_3_Text_Dataset_Cleaner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Introduction

This in-class project asks you to build a **Text Dataset Cleaner and Analyzer** — a preprocessing pipeline that takes raw, messy text entries (the kind you would get from a real-world CSV or scraped dataset) and cleans, filters, deduplicates, and analyzes them.

This is a **standalone project**. There are no hints or step-by-step scaffolding. You are given a detailed problem statement, the expected output, and some ideas for how to approach it.

By the end of this project you should be comfortable with:

* Applying **`map`**, **`filter`**, and **`reduce`** to process text data in a pipeline
* Writing a **decorator** to log what each cleaning step does
* Using a **generator** to stream entries one at a time (memory-efficient, like real large-dataset processing)
* Using **dictionaries and comprehensions** to count word frequencies
* Using **NumPy** to compute length statistics across a collection of strings

Please make sure to run <span style="color: red;">all cells</span> when done.


---
## Problem Statement

Build a `TextPipeline` class that accepts a list of raw text strings and runs them through a sequence of cleaning and analysis steps.

### What it must do

**Cleaning (in this order):**
1. Strip leading/trailing whitespace and lowercase every entry — use `map`
2. Filter out entries that are empty or whitespace-only — use `filter`
3. Filter out entries with fewer than `min_words` words — use `filter`
4. Remove duplicate entries while preserving the order they first appeared

**Analysis:**
- Count word frequencies across all cleaned entries — the total number of times each word appears across all entries combined. Use `reduce` from `functools` to merge per-entry word counts into one.
- Compute entry length statistics (in words) using NumPy: mean, std, min, max
- Track how many entries were removed at each stage (whitespace, too short, duplicate)

**Output:**
- A `run(raw_data)` method that runs the full pipeline and stores results
- A `report()` method that prints the full summary (see below)
- A `stream_entries(self)` **generator** that yields one cleaned entry at a time
- A `__str__` that gives a one-line summary
- A `__len__` that returns the number of cleaned entries

### What a complete run should look like

```python
raw_data = [
    "  Hello world  ",
    "MACHINE learning is FUN",
    "hello world",
    "",
    "AI",
    "data science and machine learning go hand in hand",
    "Python is great for data science!!",
    "   ",
    "machine learning is fun",
]

pipeline = TextPipeline(min_words=3)
pipeline.run(raw_data)

print(pipeline)
print()
pipeline.report()
print()
print("Streaming first 3 entries:")
for i, entry in enumerate(pipeline.stream_entries(), 1):
    print(f"  → {entry}")
    if i == 3:
        break
```

```
TextPipeline | 4 cleaned entries | min_words=3

=== Text Pipeline Report ===
Original entries    : 9
After cleaning      : 4
  Removed (empty)   : 2
  Removed (too short): 2
  Removed (duplicate): 1

Top 5 words:
  machine    →  3
  learning   →  3
  data       →  2
  science    →  2
  is         →  2

Length stats (words per entry):
  Mean : 6.50
  Std  : 1.73
  Min  : 5    Max : 9

Streaming first 3 entries:
  → machine learning is fun
  → data science and machine learning go hand in hand
  → python is great for data science!!
```


---
## Ideas for How to Go About It

You are not required to follow this approach — it is just one way to think through the problem.

**Plan your pipeline stages first.** Before writing the class, think about the order of operations. You clean first (strip + lowercase), then filter empties, then filter too-short entries, then deduplicate. Each stage takes a list and returns a list. Only after all cleaning do you analyze.

**Use `map` for cleaning.** `map(lambda s: s.strip().lower(), entries)` applies the transformation to every element at once. Wrap in `list()` to get a list back.

**Use `filter` for removing unwanted entries.** Two separate `filter` calls — one for empty/whitespace strings, one for entries below `min_words`. Keep track of the counts before and after each filter so you can report how many were removed.

**Deduplication with order preserved.** A plain `set()` removes duplicates but loses order. Instead, loop through the list and use a `set` just for tracking which entries you have already seen — add to the result list only if the entry is not in the seen set yet. Alternatively, `dict.fromkeys(entries)` preserves insertion order in Python 3.7+.

**Word frequency with `reduce`.** For each entry, build a per-entry frequency dict using a comprehension or a loop. Then use `reduce` to merge all those dicts into one combined dict — for each word in the new dict, add its count to the accumulator.

**Length stats with NumPy.** Convert word counts per entry into a NumPy array: `np.array([len(e.split()) for e in cleaned])`. Then use `np.mean()`, `np.std()`, `np.min()`, `np.max()`.

**The generator is simple.** `stream_entries` just loops over `self.cleaned_entries` and `yield`s one at a time. The value of a generator is that it does not load everything into memory — important for large datasets in real AI workflows.

**The `@log_step` decorator (bonus).** Write a decorator that prints the method name and the number of entries before and after the call. Apply it to your `_clean()`, `_filter_empty()`, and `_filter_short()` helper methods. This mirrors how real ML pipelines log each preprocessing step.

**Top N words.** After building the combined frequency dict, sort its items by value (descending) and take the first 5. `sorted(freq.items(), key=lambda x: x[1], reverse=True)[:5]` will do it.

**Bonus ideas if you finish early:**
- Add a `most_common_bigrams(n=5)` method that finds the most frequent two-word pairs across all entries
- Add a `filter_stopwords(stopwords)` method that removes common words like `"is"`, `"the"`, `"and"` from the frequency count
- Make the pipeline chainable — have each method return `self` so you can write `pipeline.run(data).report()`


---
## Your Code


In [1]:
import numpy as np
from functools import reduce
from collections import Counter


class TextPipeline:

    def __init__(self, min_words=1):
        self.min_words = min_words
        self.raw_count = 0
        self.cleaned_entries = []
        self.removed_empty = 0
        self.removed_short = 0
        self.removed_dup = 0
        self.word_freq = {}
        self.stats = {}

    def run(self, raw_data):
        self.raw_count = len(raw_data)

        # 1. Strip whitespace & lowercase using map
        mapped = list(map(lambda s: s.strip().lower(), raw_data))

        # 2. Filter empty entries using filter
        non_empty = list(filter(lambda s: bool(s), mapped))
        self.removed_empty = self.raw_count - len(non_empty)

        # 3. Filter short entries using filter
        valid_length = list(
            filter(
                lambda s: len(s.split()) >= self.min_words, non_empty
            )
        )
        self.removed_short = len(non_empty) - len(valid_length)

        # 4. Remove duplicates preserving order
        seen = set()
        deduped = []
        for entry in valid_length:
            if entry not in seen:
                seen.add(entry)
                deduped.append(entry)
        self.removed_dup = len(valid_length) - len(deduped)

        self.cleaned_entries = deduped

        # Analysis: Word frequencies using reduce
        def count_words(text):
            return dict(Counter(text.split()))

        def merge_counts(d1, d2):
            res = d1.copy()
            for word, count in d2.items():
                res[word] = res.get(word, 0) + count
            return res

        per_entry_counts = [count_words(entry) for entry in self.cleaned_entries]
        if per_entry_counts:
            self.word_freq = reduce(merge_counts, per_entry_counts)
        else:
            self.word_freq = {}

        # Analysis: Length statistics using NumPy
        lengths = np.array([len(e.split()) for e in self.cleaned_entries])
        if len(lengths) > 0:
            self.stats = {
                "mean": np.mean(lengths),
                "std": np.std(lengths),
                "min": np.min(lengths),
                "max": np.max(lengths),
            }
        else:
            self.stats = {"mean": 0.0, "std": 0.0, "min": 0, "max": 0}

        return self

    def stream_entries(self):
        for entry in self.cleaned_entries:
            yield entry

    def report(self):
        print("=== Text Pipeline Report ===")
        print(f"Original entries    : {self.raw_count}")
        print(f"After cleaning     : {len(self.cleaned_entries)}")
        print(f"  Removed (empty)   : {self.removed_empty}")
        print(f"  Removed (too short): {self.removed_short}")
        print(f"  Removed (duplicate): {self.removed_dup}")
        print("Top 5 words:")

        top_5 = sorted(self.word_freq.items(), key=lambda x: x[1], reverse=True)[:5]
        for word, count in top_5:
            print(f"  {word:<10} →  {count}")

        print("Length stats (words per entry):")
        print(
            f"  Mean : {self.stats.get('mean', 0):.2f}\n"
            f"  Std  : {self.stats.get('std', 0):.2f}\n"
            f"  Min  : {self.stats.get('min', 0)}    "
            f"Max : {self.stats.get('max', 0)}"
        )

    def __str__(self):
        return f"TextPipeline | {len(self.cleaned_entries)} cleaned entries | min_words={self.min_words}"

    def __len__(self):
        return len(self.cleaned_entries)

---
## Test Your Implementation

Run the full test below. If your output matches the expected report above, you are done.


In [2]:
raw_data = [
    "  Hello world  ",
    "MACHINE learning is FUN",
    "hello world",
    "",
    "AI",
    "data science and machine learning go hand in hand",
    "Python is great for data science!!",
    "   ",
    "machine learning is fun",
]

pipeline = TextPipeline(min_words=3)
pipeline.run(raw_data)

print(pipeline)
print()
pipeline.report()
print()
print("Streaming first 3 entries:")
for i, entry in enumerate(pipeline.stream_entries(), 1):
    print(f"  → {entry}")
    if i == 3:
        break


TextPipeline | 3 cleaned entries | min_words=3

=== Text Pipeline Report ===
Original entries    : 9
After cleaning     : 3
  Removed (empty)   : 2
  Removed (too short): 3
  Removed (duplicate): 1
Top 5 words:
  machine    →  2
  learning   →  2
  is         →  2
  data       →  2
  hand       →  2
Length stats (words per entry):
  Mean : 6.33
  Std  : 2.05
  Min  : 4    Max : 9

Streaming first 3 entries:
  → machine learning is fun
  → data science and machine learning go hand in hand
  → python is great for data science!!
